In [4]:
!pip install faker

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.0 MB 1.6 MB/s eta 0:00:01
   --------------------- ------------------ 1.0/2.0 MB 2.5 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 3.1 MB/s eta 0:00:00


In [4]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import timedelta

fake = Faker()
random.seed(42)
np.random.seed(42)

In [5]:
states = [
    'Abia', 'Adamawa', 'Akwa Ibom', 'Anambra', 'Bauchi', 'Bayelsa', 
    'Benue', 'Borno', 'Cross River', 'Delta', 'Ebonyi', 'Edo', 
    'Ekiti', 'Enugu', 'FCT - Abuja', 'Gombe', 'Imo', 'Jigawa', 
    'Kaduna', 'Kano', 'Katsina', 'Kebbi', 'Kogi', 'Kwara', 
    'Lagos', 'Nasarawa', 'Niger', 'Ogun', 'Ondo', 'Osun', 
    'Oyo', 'Plateau', 'Rivers', 'Sokoto', 'Taraba', 'Yobe', 'Zamfara'
]

transaction_categories = [
    'Bank Transfer',
    'P2P Transfer',
    'Airtime Top-up',
    'Data Bundle',
    'Electricity Bill',
    'Cable TV',
    'Betting Funding',
    'Merchant Payment',
    'Savings Deposit',
    'Loan Disbursement',
    'Loan Repayment',
    'POS Withdrawal',
    'Internet Service'
]


transaction_weights = {
    'Active':  [0.15, 0.12, 0.08, 0.06, 0.08, 0.07, 0.06, 0.08, 0.12, 0.05, 0.05, 0.04, 0.04],
    'At-Risk': [0.10, 0.10, 0.12, 0.10, 0.10, 0.08, 0.08, 0.07, 0.07, 0.05, 0.05, 0.05, 0.03],
    'Churned': [0.05, 0.05, 0.20, 0.18, 0.12, 0.10, 0.10, 0.08, 0.03, 0.03, 0.03, 0.02, 0.01],
    'New':     [0.08, 0.10, 0.18, 0.15, 0.12, 0.08, 0.08, 0.07, 0.05, 0.03, 0.03, 0.02, 0.01]
}

# High-revenue states (Lagos, Rivers, Abuja dominate in Nigeria)
state_weights = {
    'Lagos': 0.18, 'FCT - Abuja': 0.12, 'Rivers': 0.10,
    'Kano': 0.07, 'Oyo': 0.06, 'Delta': 0.05,
    'Anambra': 0.04, 'Kaduna': 0.04, 'Ogun': 0.04,
    'Enugu': 0.03
}
# All other states share the remaining weight evenly
remaining_states = [s for s in states if s not in state_weights]
remaining_weight = (1 - sum(state_weights.values())) / len(remaining_states)
for s in remaining_states:
    state_weights[s] = remaining_weight

### Configuring user behaviour

In [6]:
# Each status has its own behavioral profile
behavior = {
    'Active': {
        'transactions':       (30, 50),
        'amount':             (100000, 500000),
        'days_since_active':  (1, 89),       
        'engagement_days':    (300, 1825),   
        'weight':             0.25
    },
    'At-Risk': {
        'transactions':       (10, 29),
        'amount':             (30000, 150000),
        'days_since_active':  (90, 180),     
        'engagement_days':    (120, 600),
        'weight':             0.25
    },
    'Churned': {
        'transactions':       (1, 8),
        'amount':             (500, 30000),
        'days_since_active':  (181, 900),    
        'engagement_days':    (7, 120),      
        'weight':             0.30
    },
    'New': {
        'transactions':       (1, 4),
        'amount':             (500, 80000),
        'days_since_active':  (1, 30),       
        'engagement_days':    (1, 60),
        'weight':             0.20
    }
}

### Revenue (2% vs. 5%)

In [7]:
def get_charges(amount):
    if amount > 10000:
        return round(amount * 0.05, 2)   
    else:
        return round(amount * 0.02, 2)   

In [8]:
from datetime import date, timedelta
import datetime

statuses = list(behavior.keys())
status_weights = [behavior[s]['weight'] for s in statuses]

state_list = list(state_weights.keys())
state_prob  = list(state_weights.values())

data = []

for i in range(1, 1001):

    # 1. Pick status based on weight
    status = random.choices(statuses, weights=status_weights, k=1)[0]
    b = behavior[status]

    # 2. Registration date — older users for churned/active, recent for new
    if status == 'Active':
        reg_date = fake.date_between(start_date='-5y', end_date='-1y')
    elif status == 'At-Risk':
        reg_date = fake.date_between(start_date='-4y', end_date='-6m')
    elif status == 'Churned':
        reg_date = fake.date_between(start_date='-5y', end_date='-2y')
    else:  
        reg_date = fake.date_between(start_date='-2m', end_date='today')

    # 3. Last active date derived from days_since_active
    days_since = random.randint(*b['days_since_active'])
    last_active = date.today() - timedelta(days=days_since)

    # Make sure last_active is never before reg_date
    if last_active < reg_date:
        last_active = reg_date + timedelta(days=random.randint(1, 30))

    # 4. Transaction details
    num_transactions = random.randint(*b['transactions'])
    amount           = random.randint(*b['amount'])

    # Add noise — occasional outliers in every segment (realistic)
    if random.random() < 0.05:   # 5% chance of outlier
        amount = random.randint(200000, 500000)

    revenue = get_charges(amount)

    # 5. Location — weighted toward commercial hubs
    location = random.choices(state_list, weights=state_prob, k=1)[0]

    # 6. Transaction type — weighted by status behavior
    tx_type = random.choices(
        transaction_categories,
        weights=transaction_weights[status],
        k=1
    )[0]

    data.append({
        'User_ID':                f'PLN-{i:04d}',
        'Registration_Date':      reg_date,
        'Last_Active_Date':       last_active,
        'User_Location':          location,
        'Customer_Status':        status,
        'Transaction_Type':       tx_type,
        'Number_of_Transactions': num_transactions,
        'Transaction_Amount':     amount,
        'Revenue_Generated':      revenue,
    })

df = pd.DataFrame(data)
print(f"Dataset shape: {df.shape}")
print(f"\nStatus distribution:\n{df['Customer_Status'].value_counts()}")
print(f"\nLocation sample (top 5):\n{df['User_Location'].value_counts().head()}")
df.head(10)

Dataset shape: (1000, 9)

Status distribution:
Customer_Status
Churned    309
Active     266
At-Risk    240
New        185
Name: count, dtype: int64

Location sample (top 5):
User_Location
Lagos          169
FCT - Abuja    127
Rivers          97
Kano            77
Oyo             65
Name: count, dtype: int64


,User_ID,Registration_Date,Last_Active_Date,User_Location,Customer_Status,Transaction_Type,Number_of_Transactions,Transaction_Amount,Revenue_Generated
0,PLN-0001,2022-11-10,2025-09-10,Abia,Churned,Cable TV,5,8524,170.48
1,PLN-0002,2026-04-03,2026-04-22,FCT - Abuja,New,Electricity Bill,4,314629,15731.45
2,PLN-0003,2023-04-22,2023-12-10,FCT - Abuja,Churned,Merchant Payment,7,7723,154.46
3,PLN-0004,2022-10-09,2025-04-25,FCT - Abuja,Churned,Betting Funding,7,11649,582.45
4,PLN-0005,2023-11-08,2026-02-14,Anambra,Active,Savings Deposit,33,288208,14410.40
5,PLN-0006,2023-11-03,2024-04-04,FCT - Abuja,Churned,Cable TV,2,12903,645.15
6,PLN-0007,2026-04-03,2026-04-22,Bayelsa,New,POS Withdrawal,2,319484,15974.20
7,PLN-0008,2026-04-03,2026-04-07,Ekiti,New,P2P Transfer,4,36934,1846.70
8,PLN-0009,2024-05-25,2025-10-11,Kaduna,At-Risk,Betting Funding,18,121988,6099.40
9,PLN-0010,2021-10-16,2026-03-03,Yobe,Active,Merchant Payment,35,342357,17117.85


In [10]:
df.to_csv('paylink_nigeria.csv', index=False)
print("✅ Dataset saved as paylink_nigeria.csv")

✅ Dataset saved as paylink_nigeria.csv
